# HiGAN-Frequency — Kaggle training
Train only on the released SpiS-GAN train split, then evaluate `last.pth` once on the test split.

In [ ]:
# Edit only this cell.
REPO_URL = 'https://github.com/lenhanbo/HiGAN-Frequency.git'
GIT_BRANCH = 'main'
REPO_DIR = '/kaggle/working/HiGAN-Frequency'
DATA_ROOT = '/kaggle/working/spisgan_data'
OUTPUT_DIR = '/kaggle/working/higan_frequency_output'
GENERATOR_VARIANT = 'stylegan2'
EPOCHS = 70
MAX_HOURS = 4.0
CUTOFF = 0.06
PRECOMPUTE_FREQUENCY = True
PRECOMPUTE_BATCH_SIZE = 64
STYLE_RECONSTRUCTION = True
SAVE_EVERY_EPOCHS = 5
PREVIEW_EVERY_STEPS = 2000
PREVIEW_ITEMS = 16


In [ ]:
# Clone or fast-forward the selected branch.
![ -d "{REPO_DIR}/.git" ] && git -C "{REPO_DIR}" checkout "{GIT_BRANCH}" && git -C "{REPO_DIR}" pull --ff-only origin "{GIT_BRANCH}" || git clone --branch "{GIT_BRANCH}" "{REPO_URL}" "{REPO_DIR}"
%cd {REPO_DIR}
!python -m pip install -q -r requirements-kaggle.txt
!git rev-parse HEAD
from pathlib import Path
import yaml
config_path = Path('configs/config.yaml')
config = yaml.safe_load(config_path.read_text())
config['architecture']['generator_variant'] = str(GENERATOR_VARIANT)
config['training']['epochs'] = int(EPOCHS)
config['training']['max_hours'] = float(MAX_HOURS)
config['training']['style_reconstruction'] = bool(STYLE_RECONSTRUCTION)
config['training']['save_every_epochs'] = int(SAVE_EVERY_EPOCHS)
config['training']['preview_every_steps'] = int(PREVIEW_EVERY_STEPS)
config['training']['preview_items'] = int(PREVIEW_ITEMS)
config['frequency']['cutoff'] = float(CUTOFF)
config['frequency']['precompute_train'] = bool(PRECOMPUTE_FREQUENCY)
config['frequency']['precompute_batch_size'] = int(PRECOMPUTE_BATCH_SIZE)
config_path.write_text(yaml.safe_dump(config, sort_keys=False))


In [ ]:
# Download the official files and verify SHA-256, schema, image/writer counts.
!python scripts/download_spisgan.py --output-dir "{DATA_ROOT}" --repo-id DAIR-Group/SpiS-GAN
from lib.alphabet import Alphabets
assert len(Alphabets['spisgan_iam']) == 81
print('alphabet spisgan_iam:', len(Alphabets['spisgan_iam']), 'classes')


In [ ]:
# Visual smoke test: one real training image through the Gaussian FFT split.
import matplotlib.pyplot as plt
import torch
from higan_frequency.data import SpiSGANHDF5Dataset, collate_words
from higan_frequency.frequency import split_frequency_bands
preview_dataset = SpiSGANHDF5Dataset(DATA_ROOT, 'train')
preview_batch = collate_words([preview_dataset[0]])
preview_image, preview_length = preview_batch[0], preview_batch[1]
preview_low, preview_high = split_frequency_bands(preview_image, preview_length, cutoff=CUTOFF)
valid_width = int(preview_length[0])
bands = [preview_low[0, 0, :, :valid_width], preview_image[0, 0, :, :valid_width], preview_high[0, 0, :, :valid_width]]
fig, axes = plt.subplots(1, 3, figsize=(15, 3))
for axis, band, title in zip(axes, bands, ['Low frequency', 'Original', 'High frequency']):
    axis.imshow(band.detach().cpu(), cmap='gray', vmin=-1, vmax=1)
    axis.set_title(title)
    axis.axis('off')
plt.suptitle(f'Gaussian FFT split, cutoff={CUTOFF} cycles/pixel; max reconstruction error='
             f'{(bands[0] + bands[2] - bands[1]).abs().max().item():.2e}')
plt.tight_layout()
plt.show()
preview_dataset.close()


In [ ]:
# Unit checks cover dataset collation, FFT reconstruction/padding/gradient,
# shared encoder order, Generator slot mapping, and workflow isolation.
!python -m compileall -q .
!python -m pytest -q
# One real backward pass through the complete original HiGAN loss surface.
# Smoke mode deliberately skips the full 66,603-image frequency precompute.
!python train.py --config configs/config.yaml --dataset-root "{DATA_ROOT}" --output /kaggle/working/higan_frequency_smoke --smoke-test


In [ ]:
# Long training. train.py does not open test_32.hdf5 until all epochs finish;
# it then loads last.pth and runs full-test FID/KID exactly once.
!python train.py --config configs/config.yaml --dataset-root "{DATA_ROOT}" --output "{OUTPUT_DIR}"


In [ ]:
from IPython.display import Image, display
import json
output = Path(OUTPUT_DIR)
print('checkpoints:', [path.name for path in sorted((output / 'checkpoints').glob('*.pth'))])
cache_metadata = output / 'frequency_cache' / 'train_32_low_float32.json'
if cache_metadata.exists():
    print('frequency cache:', json.dumps(json.loads(cache_metadata.read_text()), indent=2))
training_previews = sorted((output / 'previews').glob('step_*.png'))
print('training previews:', len(training_previews))
if training_previews:
    display(Image(filename=str(training_previews[-1])))
metrics = json.loads((output / 'evaluation.json').read_text())
print(json.dumps(metrics, indent=2))
display(Image(filename=str(output / 'preview.png')))
